In [4]:
# ============================================================
# TESTING THE FOOD/NON-FOOD CLASSIFIER
# ============================================================

!pip install ultralytics torchvision --quiet

from google.colab import files
import torch
from torchvision import transforms, models
from PIL import Image
import urllib.request
import json

# ============================================================
# FOOD CLASSIFIER (ResNet18 pretrained on ImageNet)
# ============================================================

class FoodClassifier:
    def __init__(self):
        print("Loading pretrained image classifier...")
        self.model = models.resnet18(pretrained=True)
        self.model.eval()

        # Food-related keywords to detect
        self.food_keywords = [
            'pizza', 'cheeseburger', 'hotdog', 'burrito', 'taco', 'sandwich',
            'bagel', 'croissant', 'doughnut', 'pretzel', 'pancake', 'waffle',
            'omelet', 'custard', 'ice cream', 'cake', 'cupcake', 'cookie',
            'chocolate', 'pie', 'bread', 'loaf', 'banana', 'apple', 'orange',
            'strawberry', 'grape', 'pineapple', 'mushroom', 'broccoli', 'carrot',
            'cucumber', 'lettuce', 'tomato', 'potato', 'rice', 'pasta', 'soup',
            'salad', 'curry', 'dish', 'meal', 'plate', 'dinner', 'lunch',
            'breakfast', 'food', 'snack', 'dessert', 'fruit', 'vegetable',
            'noodle', 'burger', 'fries', 'chicken', 'beef', 'fish', 'egg',
            'cheese', 'butter', 'yogurt', 'milk', 'juice', 'coffee', 'tea'
        ]

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

        # Load ImageNet labels
        url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
        try:
            self.labels = json.load(urllib.request.urlopen(url))
        except:
            self.labels = [f"class_{i}" for i in range(1000)]

        print("✅ Food classifier ready!")

    def predict(self, image_path, verbose=True):
        """Returns (is_food, confidence, top_predictions)"""
        try:
            img = Image.open(image_path).convert('RGB')
            img_tensor = self.transform(img).unsqueeze(0)

            with torch.no_grad():
                outputs = self.model(img_tensor)
                probs = torch.softmax(outputs, dim=1)

            top_probs, top_indices = torch.topk(probs, 5)

            predictions = []
            is_food = False
            best_food_conf = 0

            for i in range(5):
                class_idx = top_indices[0][i].item()
                class_name = self.labels[class_idx].lower()
                confidence = top_probs[0][i].item()
                predictions.append((class_name, confidence))

                # Check if this class is food-related
                for keyword in self.food_keywords:
                    if keyword in class_name:
                        is_food = True
                        best_food_conf = max(best_food_conf, confidence)

            if verbose:
                print(f"   Top predictions:")
                for name, conf in predictions:
                    print(f"      {name}: {conf:.3f}")
                if is_food:
                    print(f"   ✅ Result: FOOD detected (confidence: {best_food_conf:.3f})")
                else:
                    print(f"   ❌ Result: NOT FOOD")

            return is_food, best_food_conf, predictions

        except Exception as e:
            print(f"Error: {e}")
            return False, 0, []


# ============================================================
# RUN THE TEST
# ============================================================

print("="*60)
print("FOOD/NON-FOOD CLASSIFIER TESTING")
print("="*60)

classifier = FoodClassifier()

print("\n📤 Upload images to test (upload both food and non-food images):")
print("   - Food images: donut, cake, rice, curry, etc.")
print("   - Non-food images: house, car, landscape, office, etc.")
uploaded = files.upload()

print("\n" + "="*60)
print("TEST RESULTS")
print("="*60)

results = []
for filename in uploaded.keys():
    save_path = f"test_{filename}"
    with open(save_path, 'wb') as f:
        f.write(uploaded[filename])

    print(f"\n📸 Image: {filename}")
    is_food, confidence, predictions = classifier.predict(save_path)

    results.append({
        'Image': filename,
        'Is Food': 'Yes' if is_food else 'No',
        'Confidence': f"{confidence:.3f}"
    })

# Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

food_count = sum(1 for r in results if r['Is Food'] == 'Yes')
non_food_count = len(results) - food_count

print(f"Total images tested: {len(results)}")
print(f"Classified as FOOD: {food_count}")
print(f"Classified as NON-FOOD: {non_food_count}")

print("\nDetailed Results:")
for r in results:
    print(f"   {r['Image']}: {r['Is Food']} (conf: {r['Confidence']})")

# Save results
import pandas as pd
df = pd.DataFrame(results)
df.to_csv('classifier_test_results.csv', index=False)
print("\n✅ Results saved to 'classifier_test_results.csv'")

FOOD/NON-FOOD CLASSIFIER TESTING
Loading pretrained image classifier...
✅ Food classifier ready!

📤 Upload images to test (upload both food and non-food images):
   - Food images: donut, cake, rice, curry, etc.
   - Non-food images: house, car, landscape, office, etc.


Saving istockphoto-639414496-612x612.jpg to istockphoto-639414496-612x612 (1).jpg
Saving istockphoto-1588965233-612x612.jpg to istockphoto-1588965233-612x612 (1).jpg
Saving test1.jpg to test1 (2).jpg
Saving test2.jpg to test2 (2).jpg

TEST RESULTS

📸 Image: istockphoto-639414496-612x612 (1).jpg
   Top predictions:
      patio: 0.535
      mobile home: 0.155
      maze: 0.052
      lakeshore: 0.036
      window screen: 0.035
   ❌ Result: NOT FOOD

📸 Image: istockphoto-1588965233-612x612 (1).jpg
   Top predictions:
      window shade: 0.202
      patio: 0.156
      couch: 0.146
      entertainment center: 0.098
      sliding door: 0.089
   ❌ Result: NOT FOOD

📸 Image: test1 (2).jpg
   Top predictions:
      carbonara: 0.336
      plate: 0.319
      burrito: 0.057
      mashed potato: 0.040
      hot dog: 0.030
   ✅ Result: FOOD detected (confidence: 0.319)

📸 Image: test2 (2).jpg
   Top predictions:
      bakery: 0.201
      hot dog: 0.096
      paper towel: 0.075
      chocolate syrup: 